In [2]:
print('Lets Start')

Lets Start


In [3]:
import re
import string
import automac
import mlflow
import mlflow.sklearn
import dagshub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score 
from sklearn.feature_extraction.text import CountVectorizer

[nltk_data] Error loading stopwords: <urlopen error [Errno 101]
[nltk_data]     Network is unreachable>


In [18]:
df = pd.read_csv('/mnt/WindowsD/Artificial Intelligence/Datasets/tweets_emotions.csv' , index_col=0).drop(columns='tweet_id')
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [19]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [20]:
x=(df['sentiment']=='happiness') | (df['sentiment'] == 'sadness')
df = df[x]

In [23]:
df['sentiment'] =df['sentiment'].replace({'sadness' : 0 , 'happiness' : 1}) 

In [24]:
df

,sentiment,content
1,0,Layin n bed with a headache ughhhh...waitin o...
2,0,Funeral ceremony...gloomy friday...
6,0,"I should be sleep, but im not! thinking about ..."
8,0,@charviray Charlene my love. I miss you
9,0,@kelcouch I'm sorry at least it's Friday?
...,...,...
39986,1,going to watch boy in the striped pj's hope i ...
39987,1,"gave the bikes a thorough wash, degrease it an..."
39988,1,"had SUCH and AMAZING time last night, McFly we..."
39994,1,Succesfully following Tayla!!


In [42]:
x_ = df['content'].apply(automac.text.transform)
x_ 

1                   layin n bed headach ughhhh waitin call
2                             funer ceremoni gloomi friday
6        sleep im think old friend want marri damn amp ...
8                              charviray charlen love miss
9                              kelcouch sorri least friday
                               ...                        
39986                      go watch boy stripe pj hope cri
39987    gave bike thorough wash degreas greas think re...
39988                    amaz time last night mcfli incred
39994                                  succes follow tayla
39998    niariley wassup beauti follow peep new hit sin...
Name: content, Length: 10374, dtype: str

In [43]:
y = df['sentiment']
y

1        0
2        0
6        0
8        0
9        0
        ..
39986    1
39987    1
39988    1
39994    1
39998    1
Name: sentiment, Length: 10374, dtype: object

In [45]:
cv =CountVectorizer(max_features=1000)
x_trans = cv.fit_transform(x_)


In [46]:
x = pd.DataFrame(x_trans.toarray())

In [47]:
x_train, x_test , y_train, y_test= train_test_split(x , y, test_size=0.2 , random_state=0)

In [29]:
try:
    dagshub.init(repo_owner='jubito27', repo_name='mlops-mini-project', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/mlops-mini-project.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('LogisticRegression Baseline')


Accessing as jubito27

Initialized MLflow to track repo "jubito27/mlops-mini-project"

Repository jubito27/mlops-mini-project initialized!

2026/11/14 10:18:13 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f39e159e480640e4875ad026192a0729', creation_time=1794631694930, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794631694930, lifecycle_stage='active', name='LogisticRegression Baseline', tags={}, trace_location=None, workspace='default'>

In [48]:
with mlflow.start_run():
    mlflow.log_param("vecrtorizer" , 'BOW') 
    mlflow.log_param("num_features" , 1000) 
    mlflow.log_param("test_size" , 0.2) 

    model = LogisticRegression()
    y_train = pd.to_numeric(y_train, errors='raise').astype(int)
    y_test = pd.to_numeric(y_test, errors='raise').astype(int)
    model.fit(x_train, y_train)

    y_pred= model.predict(x_test)

    mlflow.log_metric('accuracy' , accuracy_score(y_test , y_pred))
    mlflow.log_metric('precision' , precision_score(y_test , y_pred))
    mlflow.log_metric('recall' , recall_score(y_test , y_pred))
    mlflow.log_metric('f1_score' , f1_score(y_test , y_pred))

    mlflow.sklearn.log_model(model , 'model' )

    import os
    path = '1_baseline_model.ipynb'
    os.system(f"jupyter notebbok --to notebook --execute --inplace {path}")
    print("Everything Done")

2026/11/14 10:39:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 10:40:06 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
usage: jupyter [-h] [--json] [--debug]
               [--version | --config-dir | --data-dir | --runtime-dir |
               --paths | subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show al

Everything Done
🏃 View run bustling-cub-793 at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/1/runs/6270d29f1c924809b755a50354233a67
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/1
